In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from category_encoders import BinaryEncoder
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn import set_config
from lightgbm import LGBMRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder

#### 데이터 로딩

In [ ]:
X_train = pd.read_csv('../data/X_train.csv', encoding='cp949')
y_train = pd.read_csv('../data/y_train.csv', encoding='cp949').Salary

In [ ]:
# 결측치가 있는 피처
missing_features = X_train.columns[X_train.isnull().any()].tolist()
missing_features 

In [ ]:
X_train.head()

In [ ]:
X_train.info()

- binary가 압도적으로 많음

---

In [ ]:
#score 데이터프레임 만들기
data = { "Model": [],
        "Encoder": [],
        "Mean Score": [],
        "Std Score": []
        }

score_df = pd.DataFrame(data)
score_df
score_df = pd.DataFrame(columns=["Model", "Encoder", "Mean Score", "Std Score"])

---

In [ ]:
# RandomForestRegressor

preprocessor = Pipeline(
    steps=[ 
        ("imputer", SimpleImputer(strategy="most_frequent")), 
        ("encoder",  OneHotEncoder(sparse_output=True, handle_unknown='ignore'))
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor), 
        ( "regressor", RandomForestRegressor())
    ]
)

scores = cross_val_score(model, X_train, y_train, scoring='neg_mean_squared_error', cv=5)

# 변수 정의
name = type(model["regressor"]).__name__
encoder_name = model["preprocessor"]["encoder"].__class__.__name__
score_mean = round(np.sqrt(-1 * scores.mean()), 2)  
score_std = round(np.sqrt(scores.std()), 2) 

# score_df에 값 추가
score_df.loc[len(score_df)] = [name, encoder_name, score_mean, score_std]

# 성능 출력
print(f"{name} CV mean = {score_mean} with std = {score_std}")

In [ ]:
# MLPRegressor
preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")), 
        ("encoder",  OneHotEncoder(sparse_output=True, handle_unknown='ignore')),
    ]
)

model2_MLP = Pipeline(
    steps=[
        ("preprocessor", preprocessor), 
        ( "regressor", MLPRegressor())
    ]
)

name = type(model2_MLP["regressor"]).__name__
encoder_name = model2_MLP["preprocessor"]["encoder"].__class__.__name__  #추가
score_mean = round(np.sqrt(-1 * scores.mean()), 2)  #추가
score_std = round(np.sqrt(scores.std()), 2)   #추가

scores = cross_val_score(model2_MLP, X_train, y_train, scoring='neg_mean_squared_error', cv=5)


print(f"{name} CV mean = {score_mean} with std = {score_std}") #변수명으로만 바꿈

In [ ]:
score_df